In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler


In [2]:
df = pd.read_csv(r"C:\AI-Predictive-Maintenance\memory\out.csv")

In [3]:
df.dropna()

,id,ts,host_id,cpu_usage_pct,memory_usage_pct,power_kw,status,rolling_mean_1h,rolling_mean_24h,rolling_std_1h,rolling_std_24h,growth_rate,acceleration,Z_score,trend,volatility_ratio
21,64,2026-04-02 13:03:08.404+00,1,4.00,75.00,0.221,Normal,74.045455,74.045455,0.213201,0.213201,0.013514,0.013514,4.477215,0.000000,0.000000
22,67,2026-04-02 13:05:08.513+00,1,4.00,74.00,0.220,Normal,74.043478,74.043478,0.208514,0.208514,-0.013333,-0.026847,-0.208514,0.000000,0.000000
23,70,2026-04-02 13:07:08.512+00,1,6.00,74.00,0.220,Normal,74.041667,74.041667,0.204124,0.204124,0.000000,0.013333,-0.204124,0.000000,0.000000
24,73,2026-04-02 13:09:08.403+00,1,4.00,74.00,0.221,Normal,74.040000,74.040000,0.200000,0.200000,0.000000,0.000000,-0.200000,0.000000,0.000000
25,76,2026-04-02 13:11:08.405+00,1,4.00,74.00,0.220,Normal,74.038462,74.038462,0.196116,0.196116,0.000000,0.000000,-0.196116,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24757,74272,2026-06-11 04:06:37.22+00,1,3.83,81.75,0.220,Warning,81.750769,81.652414,0.002774,0.042917,0.000000,0.000122,2.273814,0.098355,-0.040144
24758,74275,2026-06-11 04:11:36.52+00,1,4.94,81.76,0.220,Warning,81.751667,81.653000,0.003892,0.043222,0.000122,0.000122,2.475598,0.098667,-0.039329
24759,74278,2026-06-11 04:16:38.55+00,1,6.28,81.76,0.220,Warning,81.752500,81.653552,0.004523,0.043563,0.000000,-0.000122,2.443556,0.098948,-0.039040
24760,74281,2026-06-11 04:21:38.949+00,1,5.76,81.76,0.221,Warning,81.753333,81.654138,0.004924,0.043848,0.000000,0.000000,2.414319,0.099195,-0.038924


In [4]:
df.columns

Index(['id', 'ts', 'host_id', 'cpu_usage_pct', 'memory_usage_pct', 'power_kw',
       'status', 'rolling_mean_1h', 'rolling_mean_24h', 'rolling_std_1h',
       'rolling_std_24h', 'growth_rate', 'acceleration', 'Z_score', 'trend',
       'volatility_ratio'],
      dtype='object')

In [5]:
df.drop(columns=['host_id', 'power_kw', 'status'], axis=0, inplace=True)
df

,id,ts,cpu_usage_pct,memory_usage_pct,rolling_mean_1h,rolling_mean_24h,rolling_std_1h,rolling_std_24h,growth_rate,acceleration,Z_score,trend,volatility_ratio
0,1,2026-04-02 12:22:07.517+00,4.00,74.00,74.000000,74.000000,NaN,NaN,NaN,NaN,NaN,0.000000,NaN
1,4,2026-04-02 12:24:02.882+00,5.00,74.00,74.000000,74.000000,0.000000,0.000000,0.000000,NaN,NaN,0.000000,0.000000
2,7,2026-04-02 12:26:01.7+00,4.00,74.00,74.000000,74.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.000000
3,10,2026-04-02 12:28:01.845+00,7.00,74.00,74.000000,74.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.000000
4,13,2026-04-02 12:30:02.091+00,5.00,74.00,74.000000,74.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
24757,74272,2026-06-11 04:06:37.22+00,3.83,81.75,81.750769,81.652414,0.002774,0.042917,0.000000,0.000122,2.273814,0.098355,-0.040144
24758,74275,2026-06-11 04:11:36.52+00,4.94,81.76,81.751667,81.653000,0.003892,0.043222,0.000122,0.000122,2.475598,0.098667,-0.039329
24759,74278,2026-06-11 04:16:38.55+00,6.28,81.76,81.752500,81.653552,0.004523,0.043563,0.000000,-0.000122,2.443556,0.098948,-0.039040
24760,74281,2026-06-11 04:21:38.949+00,5.76,81.76,81.753333,81.654138,0.004924,0.043848,0.000000,0.000000,2.414319,0.099195,-0.038924


In [6]:
df['ts'] = pd.to_datetime(df['ts'], format='mixed').dt.tz_localize(None)

In [7]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error


In [8]:
df = df.sort_values('ts')

leakage_cols = ['rolling_mean_1h', 'rolling_mean_24h', 'rolling_std_1h', 'rolling_std_24h', 'growth_rate', 'acceleration', 'Z_score', 'trend', 'volatility_ratio']
df[leakage_cols] = df[leakage_cols].shift(1)


In [9]:
FORECAST_HORIZON = 12

df['target_forecast'] = df['memory_usage_pct'].shift(-FORECAST_HORIZON)


In [10]:
features = df.drop(columns=['id', 'ts', 'target_forecast'])
target=df['target_forecast']

X_temp, X_test, y_temp, y_test = train_test_split(features, target, test_size=0.15, shuffle=False)

val_size = 0.10/0.85

X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=val_size, shuffle=False)

print(f"Data shapes - Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")


Data shapes - Train: (18570, 11), Val: (2477, 11), Test: (3715, 11)


In [11]:
model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_states=42
)

model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)


c:\AI-Predictive-Maintenance\venv\lib\site-packages\xgboost\training.py:200: UserWarning: [09:32:18] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "random_states" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [12]:
#Predictions
y_pred_train = model.predict(X_train)
y_pred_val = model.predict(X_val)
y_pred_test = model.predict(X_test)

In [13]:

def print_metrics(name, y_true, y_pred):
    print(f"--- {name} Metrics ---")
    print(f"RMSE: {mean_squared_error(y_true, y_pred) ** 0.5:.4f}")
    print(f"MAE:  {mean_absolute_error(y_true, y_pred):.4f}")
    print(f"R2:   {r2_score(y_true, y_pred):.4f}")
    print(f"MAPE: {mean_absolute_percentage_error(y_true, y_pred):.4f}\n")

print_metrics("Train", y_train, y_pred_train)
print_metrics("Validation", y_val, y_pred_val)
valid_idx = ~y_test.isna()
print_metrics("Test", y_test[valid_idx], y_pred_test[valid_idx])


--- Train Metrics ---
RMSE: 1.0875
MAE:  0.1252
R2:   0.9980
MAPE: 0.0042

--- Validation Metrics ---
RMSE: 6.4409
MAE:  3.9565
R2:   -59.8900
MAPE: 0.1369

--- Test Metrics ---
RMSE: 6.4528
MAE:  4.5721
R2:   0.9365
MAPE: 0.0837



In [14]:
#features for the anomaly detection model
features = ['memory_usage_pct', 'rolling_std_24h', 'growth_rate', 'acceleration']
X = df[features]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
    

In [15]:
#training of the isolation forest model
from sklearn.ensemble import IsolationForest

model = IsolationForest(
    contamination='auto',
    random_state=42
)

model.fit(X_scaled)

,n_estimators,100
,max_samples,'auto'
,contamination,'auto'
,max_features,1.0
,bootstrap,False
,n_jobs,None
,random_state,42
,verbose,0
,warm_start,False


In [16]:
#anomaly detection for server 1
scores = model.decision_function(X_scaled)

df['anomaly_score'] = -scores


In [17]:
df.head()

,id,ts,cpu_usage_pct,memory_usage_pct,rolling_mean_1h,rolling_mean_24h,rolling_std_1h,rolling_std_24h,growth_rate,acceleration,Z_score,trend,volatility_ratio,target_forecast,anomaly_score
0,1,2026-04-02 12:22:07.517,4.0,74.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,74.0,-0.144829
1,4,2026-04-02 12:24:02.882,5.0,74.0,74.0,74.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,74.0,-0.144829
2,7,2026-04-02 12:26:01.700,4.0,74.0,74.0,74.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,74.0,-0.131572
3,10,2026-04-02 12:28:01.845,7.0,74.0,74.0,74.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,74.0,-0.131572
4,13,2026-04-02 12:30:02.091,5.0,74.0,74.0,74.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,74.0,-0.131572


In [18]:
#current risk
def current_risk(memory_usage):
    if memory_usage < 50:
        return 10
    
    elif memory_usage < 70:
        return 30
    
    elif memory_usage < 86:
        return 70
    
    else:
        return 100
    
    

In [19]:
#forecast_risk
def forecast_risk(forecast_memory):

    if forecast_memory < 50:
        return 10

    elif forecast_memory < 70:
        return 40

    elif forecast_memory < 85:
        return 80

    else:
        return 100

In [ ]:
#anomaly risk
import numpy as np
def anomaly_risk(score, min_score=-0.3, max_score=0.3):

    risk = 100 * ((score- min_score)/ (max_score - min_score))


    return np.clip(risk,0,100)

In [21]:
#stability_risk
max_std = df["rolling_std_24h"].max()

def stability_risk(std):

    return min(
        100,
        (std / max_std) * 100
    )

In [25]:
# risk_score = (0.35 * current_risk + 0.3 * forecast_risk + 0.20 * anomaly_risk + 0.15 * stability_risk)
df['risk_score'] = (
    0.35 * df['memory_usage_pct'].apply(current_risk) +
    0.30 * df['target_forecast'].apply(forecast_risk) +
    0.20 * df['anomaly_score'].apply(anomaly_risk) +
    0.15 * df['rolling_std_24h'].apply(stability_risk)
)

df.head()

,id,ts,cpu_usage_pct,memory_usage_pct,rolling_mean_1h,rolling_mean_24h,rolling_std_1h,rolling_std_24h,growth_rate,acceleration,Z_score,trend,volatility_ratio,target_forecast,anomaly_score,risk_score
0,1,2026-04-02 12:22:07.517,4.0,74.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,74.0,-0.144829,79.293169
1,4,2026-04-02 12:24:02.882,5.0,74.0,74.0,74.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,74.0,-0.144829,79.293169
2,7,2026-04-02 12:26:01.700,4.0,74.0,74.0,74.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,74.0,-0.131572,63.762889
3,10,2026-04-02 12:28:01.845,7.0,74.0,74.0,74.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,74.0,-0.131572,63.762889
4,13,2026-04-02 12:30:02.091,5.0,74.0,74.0,74.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,74.0,-0.131572,63.762889


In [28]:
import openpyxl
df.to_excel('result_riskscoring.xlsx')